In [1]:
%pip install psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [2]:

import psycopg2


In [3]:
# Set environment variables
HOST="localhost"
DB="postgres"
USER="postgres"
PASSWORD="postgres"
PORT=5441

In [4]:
import psycopg2
import pandas as pd

from data.db_loader import load_from_postgres
from features.features import create_features


DB_CONFIG = {
    "host": "localhost",
    "database": "postgres",
    "user": "postgres",
    "password": "postgres",
    "port": 5441,
}

TABLE_NAME = "conso_clean"

def create_table(conn):
    cur = conn.cursor()

    cur.execute(f"""
        CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
            id BIGSERIAL PRIMARY KEY,
            consommation FLOAT,
            prevision_j_1 FLOAT,
            prevision_j FLOAT,
            fioul FLOAT,
            charbon FLOAT,
            gaz FLOAT,
            nucleaire FLOAT,
            eolien FLOAT,
            solaire FLOAT,
            hydraulique FLOAT,
            pompage FLOAT,
            bioenergies FLOAT,
            ech_physiques FLOAT,
            taux_de_co2 FLOAT,
            ech_comm_angleterre FLOAT,
            ech_comm_espagne FLOAT,
            ech_comm_italie FLOAT,
            ech_comm_suisse FLOAT,
            ech_comm_allemagne_belgique FLOAT,
            fioul_tac FLOAT,
            fioul_cogen FLOAT,
            fioul_autres FLOAT,
            gaz_tac FLOAT,
            gaz_cogen FLOAT,
            gaz_ccg FLOAT,
            gaz_autres FLOAT,
            hydraulique_fil_de_leau_eclusee FLOAT,
            hydraulique_lacs FLOAT,
            hydraulique_step_turbinage FLOAT,
            bioenergies_dechets FLOAT,
            bioenergies_biomasse FLOAT,
            bioenergies_biogaz FLOAT,
            destockage_batterie FLOAT,
            eolien_terrestre FLOAT,
            eolien_offshore FLOAT,
            stockage_batterie TEXT,
            year INTEGER,
            hour INTEGER,
            day INTEGER,
            month INTEGER,
            dayofweek INTEGER,
            weekend INTEGER
        );
    """)

    conn.commit()
    cur.close()

def insert_features(conn, df: pd.DataFrame):
    cur = conn.cursor()

    cols = ",".join(df.columns)
    placeholders = ",".join(["%s"] * len(df.columns))

    query = f"""
        INSERT INTO {TABLE_NAME} ({cols})
        VALUES ({placeholders})
    """

    for _, row in df.iterrows():
        row = row.where(pd.notnull(row), None)
        cur.execute(query, tuple(row))

    conn.commit()
    cur.close()


def ingest_features():
    print(f"Chargement depuis {TABLE_NAME}")
    df_raw = load_from_postgres()

    print("Création des features")
    df_feat = create_features(df_raw)
    df_feat = df_feat.drop(columns=["id"], errors="ignore")

    print("Connexion PostgreSQL")
    conn = psycopg2.connect(**DB_CONFIG)

    print("Création table features")
    create_table(conn)

    print("Reset features table")
    cur = conn.cursor()
    cur.execute(f"TRUNCATE TABLE {TABLE_NAME}")
    conn.commit()
    cur.close()

    print("Insertion features")
    insert_features(conn, df_feat)

    conn.close()
    print("Features chargées dans PostgreSQL")


if __name__ == "__main__":
    ingest_features()


Chargement depuis conso_clean


/home/c-enjalbert/Documents/EPSI/MSPR/bloc_3/EDF-mspr3/src/data/db_loader.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM eco2mix_raw", conn)


Création des features
Connexion PostgreSQL
Création table features
Reset features table
Insertion features
Features chargées dans PostgreSQL


In [7]:
import psycopg2

DB_CONFIG = {
    "host": "localhost",
    "database": "postgres",
    "user": "postgres",
    "password": "postgres",
    "port": 5441
}

SQL = """
DROP TABLE IF EXISTS aggregated_conso_weather;

CREATE TABLE aggregated_conso_weather AS
WITH conso AS (
    SELECT
        id AS conso_id,
        consommation,
        prevision_j_1,
        prevision_j,
        fioul,
        charbon,
        gaz,
        nucleaire,
        eolien,
        solaire,
        hydraulique,
        pompage,
        bioenergies,
        ech_physiques,
        taux_de_co2,
        ech_comm_angleterre,
        ech_comm_espagne,
        ech_comm_italie,
        ech_comm_suisse,
        ech_comm_allemagne_belgique,
        fioul_tac,
        fioul_cogen,
        fioul_autres,
        gaz_tac,
        gaz_cogen,
        gaz_ccg,
        gaz_autres,
        hydraulique_fil_de_leau_eclusee,
        hydraulique_lacs,
        hydraulique_step_turbinage,
        bioenergies_dechets,
        bioenergies_biomasse,
        bioenergies_biogaz,
        destockage_batterie,
        eolien_terrestre,
        eolien_offshore,
        stockage_batterie,
        year,
        month,
        day,
        hour,
        dayofweek,
        weekend,
        make_timestamp(year, month, day, hour, 0, 0) AS datetime
    FROM conso_clean
),
weather_fr AS (
    SELECT
        datetime,
        AVG(temperature_2m)          AS temp_fr,
        AVG(relative_humidity_2m)    AS hum_fr,
        AVG(snowfall)                AS snow_fr,
        AVG(precipitation)           AS rain_fr
    FROM weather_data
    GROUP BY datetime
)
SELECT
    ROW_NUMBER() OVER (ORDER BY c.datetime) AS id,
    c.*,
    w.temp_fr,
    w.hum_fr,
    w.snow_fr,
    w.rain_fr
FROM conso c
LEFT JOIN weather_fr w
    ON c.datetime = w.datetime
ORDER BY c.datetime;
"""

def ingest_conso_meteo():
    print("Connexion PostgreSQL...")
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    cur = conn.cursor()

    print("Création de aggregated_conso_weather")
    cur.execute(SQL)

    cur.close()
    conn.close()
    print("aggregated_conso_weather créée")


In [8]:
ingest_conso_meteo()

Connexion PostgreSQL...
Création de aggregated_conso_weather
aggregated_conso_weather créée


In [6]:
table_name="test_table3"
create_table(conn, table_name)

NameError: name 'conn' is not defined

In [ ]:
conn.close()